In [10]:
# ==============================
# Task 12 : LangChain Agents
# POC1 : Source-based query : To develop 2 agents each of completely different source of information and 
# Routing the user query to it's agent and get the final answer
# ==============================

from langchain_ollama import ChatOllama
from langchain_core.tools import Tool

# ------------------------------
# 1. Load Llama3 Model
# ------------------------------
llm = ChatOllama(
    model="llama3",
    temperature=0
)

# ------------------------------
# 2. Define Agents (Tools)
# ------------------------------
def docs_agent(query: str) -> str:
    return f"[DOCS ANSWER]: Internal knowledge for -> {query}"

def web_agent(query: str) -> str:
    return f"[WEB ANSWER]: General knowledge for -> {query}"

docs_tool = Tool(
    name="DocsTool",
    func=docs_agent,
    description="Use only for internal company documents, policies, or private info"
)

web_tool = Tool(
    name="WebTool",
    func=web_agent,
    description="Use only for general knowledge, public info, or news"
)

tools = [docs_tool, web_tool]

# ------------------------------
# 3. Router Function
# ------------------------------
'''def route_query(query: str) -> str:
    prompt = f"""
You are a strict router. Decide which tool to use for the user query.
Available tools: DocsTool, WebTool

Query: {query}

Return ONLY one word: DocsTool or WebTool
"""
    decision = llm.invoke(prompt).content.strip()

    if "DocsTool" in decision:
        return docs_tool.run(query)
    elif "WebTool" in decision:
        return web_tool.run(query)
    else:
        return f" Routing failed: {decision}" '''
def route_query(query: str) -> str:
    prompt = f"""
You are a strict router. Your job is to choose the correct tool for the query.
Available tools: DocsTool (internal/company knowledge) or WebTool (general knowledge, news, public info).

Instructions:
- Respond ONLY with the tool name: DocsTool or WebTool.
- Do NOT add extra words or punctuation.
- Use DocsTool only for internal policies, company info, or private knowledge.
- Use WebTool for anything general, scientific, historical, or public knowledge.

Query: {query}
"""

    # Ask Llama3
    decision = llm.invoke(prompt).content.strip()

    # Safety: force exact match
    if "Doc" in decision:
        return docs_tool.run(query)
    elif "Web" in decision:
        return web_tool.run(query)
    else:
        return f"Routing failed (Llama3 said: {decision})"

# ------------------------------
# 4. Run CLI
# ------------------------------
if __name__ == "__main__":
    print(" Multi-Agent Router (Llama3)")
    print("Type 'exit' to quit\n")

    while True:
        user_query = input("You: ")

        if user_query.lower() == "exit":
            break

        response = route_query(user_query)
        print(f"\nAI: {response}\n")

 Multi-Agent Router (Llama3)
Type 'exit' to quit



You:  What is our company leave policy?



AI: [DOCS ANSWER]: Internal knowledge for -> What is our company leave policy?



You:  What is AI ?



AI: [WEB ANSWER]: General knowledge for -> What is AI ?



You:  Explain Newton's laws of motion?



AI: [WEB ANSWER]: General knowledge for -> Explain Newton's laws of motion?



You:  Explain the procedure for expense reimbursement?



AI: [DOCS ANSWER]: Internal knowledge for -> Explain the procedure for expense reimbursement?



You:  exit
